# ACID Transaction with a Bank Transfer

This notebook demonstrates **Atomicity, Consistency, Isolation, and Durability (ACID)** using a transfer of 200 between a Savings and a Current account in DuckDB.

Walkthrough:

1. Create the accounts and run a successful transfer.3. Simulate the same failure **with** a transaction and see it roll back cleanly.
2. Simulate a failure **without** a transaction and watch the database become inconsistent.

## 1. Install and import DuckDB

Run the installation cell if DuckDB is not already installed.

In [3]:
#%pip install -q duckdb

In [4]:
import duckdb

# Create an in-memory DuckDB connection
con = duckdb.connect()

def show_balances():
    # Show each account's balance plus a Total row
    con.sql("""
    SELECT * FROM (
        SELECT account_type, balance FROM accounts
        UNION ALL
        SELECT 'Total', SUM(balance) FROM accounts
    )
    """).show()


## 2. Create the accounts table and insert test data

In [5]:
con.execute("""
DROP TABLE IF EXISTS accounts;
""")

# account_id is the primary key; both rows below belong to the same customer
con.execute("""
CREATE TABLE accounts (
    account_id INTEGER PRIMARY KEY,
    customer_name VARCHAR,
    account_type VARCHAR,
    balance DECIMAL(10, 2)
);
""")

con.execute("""
INSERT INTO accounts (account_id, customer_name, account_type, balance)
VALUES
    (1, 'Ahmed', 'Savings', 1000.00),
    (2, 'Ahmed', 'Current', 500.00);
""")

con.sql("SELECT * FROM accounts ORDER BY account_id").show()


┌────────────┬───────────────┬──────────────┬───────────────┐
│ account_id │ customer_name │ account_type │    balance    │
│   int32    │    varchar    │   varchar    │ decimal(10,2) │
├────────────┼───────────────┼──────────────┼───────────────┤
│          1 │ Ahmed         │ Savings      │       1000.00 │
│          2 │ Ahmed         │ Current      │        500.00 │
└────────────┴───────────────┴──────────────┴───────────────┘



Expected balances after the successful transfer:

| Account | Balance |
|---|---:|
| Savings | 800.00 |
| Current | 700.00 |
| Total | 1,500.00 |

## 3. Transfer without a transaction: an exception leaves the database inconsistent

Doing the transfer **without** wrapping the two updates in a transaction. Outside an explicit transaction, DuckDB commits each statement as soon as it succeeds.

The withdrawal from Savings commits immediately. The deposit into Current then fails (nonexistent column), so the money never arrives - it simply disappears.

In [8]:
try:
    # Step 1: Withdraw 200 from Savings - commits immediately, no transaction wraps it
    con.execute("""
    UPDATE accounts
    SET balance = balance - 200
    WHERE account_id = 1;
    """)

    # Step 2: Deliberately reference a nonexistent column to trigger an error
    con.execute("""
    UPDATE accounts
    SET balance = balance + 200
    WHERE account_id = 2
      AND invalid_column = 1;
    """)
except Exception as error:
    print(f"Simulated failure: {error}")
    print("There is no transaction to roll back, so the withdrawal from step 1 stays applied.")

show_balances()

# Spell out the human cost of the inconsistency
total = con.sql("SELECT SUM(balance) FROM accounts").fetchone()[0]
print(f"\nReal-world effect: the customer's total dropped from 1,500.00 to {total:,.2f}.")
print('\U0001F622 Customer: "I moved 200 into my Current account, but it just vanished!"')
print('\U0001F600 Bank: now owes the customer 200 less than it should - an unearned gain caused by the missing update.')

Simulated failure: Binder Error: Referenced column "invalid_column" not found in FROM clause!
Candidate bindings: "account_id", "balance"

LINE 5:       AND invalid_column = 1;
                  ^
There is no transaction to roll back, so the withdrawal from step 1 stays applied.
┌──────────────┬───────────────┐
│ account_type │    balance    │
│   varchar    │ decimal(38,2) │
├──────────────┼───────────────┤
│ Savings      │        800.00 │
│ Current      │        500.00 │
│ Total        │       1300.00 │
└──────────────┴───────────────┘


Real-world effect: the customer's total dropped from 1,500.00 to 1,300.00.
😢 Customer: "I moved 200 into my Current account, but it just vanished!"
😀 Bank: now owes the customer 200 less than it should - an unearned gain caused by the missing update.


Resulting balances without a transaction:

| Account | Balance |
|---|---:|
| Savings | 800.00 |
| Current | 500.00 |
| Total | 1,300.00 |

The database is now **inconsistent**: the total dropped from 1,500.00 to 1,300.00 because the withdrawal committed but the deposit never happened, and there was no transaction to undo it.

😢 The customer's 200 is simply gone. 😀 The bank's liability toward the customer just shrank by 200 - a real loss for one side and an unearned gain for the other, caused only by skipping a transaction.

## 4. Reset the test data again

Restore the original balances before demonstrating how a transaction prevents this problem.

In [9]:
# Restore original balances so this demo starts from a clean, known state
con.execute("""
UPDATE accounts
SET balance = CASE
    WHEN account_id = 1 THEN 1000.00
    WHEN account_id = 2 THEN 500.00
END;
""")

show_balances()


┌──────────────┬───────────────┐
│ account_type │    balance    │
│   varchar    │ decimal(38,2) │
├──────────────┼───────────────┤
│ Savings      │       1000.00 │
│ Current      │        500.00 │
│ Total        │       1500.00 │
└──────────────┴───────────────┘



## 5. With a transaction: the exception leaves no partial changes

Repeat the same failing sequence, wrapped in `BEGIN TRANSACTION` / `COMMIT`. When the second update fails, the `except` block issues a `ROLLBACK`, which undoes the withdrawal too - it was never committed on its own. The two updates succeed or fail together (atomicity), so the balances can never be observed in a partially-updated, inconsistent state.

In [10]:
con.execute("BEGIN TRANSACTION;")

try:
    # Step 1: Withdraw 200 from Savings - only tentative until COMMIT
    con.execute("""
    UPDATE accounts
    SET balance = balance - 200
    WHERE account_id = 1;
    """)

    # Step 2: Deliberately reference a nonexistent column to trigger an error
    con.execute("""
    UPDATE accounts
    SET balance = balance + 200
    WHERE account_id = 2
      AND invalid_column = 1;
    """)

    con.execute("COMMIT;")
except Exception as error:
    print(f"Simulated failure: {error}")
    # Undo step 1 as well, since it was never committed on its own
    con.execute("ROLLBACK;")
    print("Transaction rolled back.")

show_balances()


Simulated failure: Binder Error: Referenced column "invalid_column" not found in FROM clause!
Candidate bindings: "account_id", "balance"

LINE 5:       AND invalid_column = 1;
                  ^
Transaction rolled back.
┌──────────────┬───────────────┐
│ account_type │    balance    │
│   varchar    │ decimal(38,2) │
├──────────────┼───────────────┤
│ Savings      │       1000.00 │
│ Current      │        500.00 │
│ Total        │       1500.00 │
└──────────────┴───────────────┘



## 8. Verify the rollback

Balances are restored to their original values - the withdrawal was undone along with the failed deposit, so the total never became inconsistent, unlike the earlier example without a transaction.

| Account | Balance |
|---|---:|
| Savings | 1,000.00 |
| Current | 500.00 |
| Total | 1,500.00 |

In [11]:
# Final check: balances plus Total, confirming the database is consistent
result = con.sql("""
SELECT * FROM (
    SELECT account_type, balance FROM accounts
    UNION ALL
    SELECT 'Total', SUM(balance) FROM accounts
)
ORDER BY CASE account_type WHEN 'Savings' THEN 1 WHEN 'Current' THEN 2 ELSE 3 END;
""").df()

display(result)


,account_type,balance
0,Savings,1000.0
1,Current,500.0
2,Total,1500.0


## ACID properties illustrated

- **Atomicity:** Both updates succeed, or neither is applied - shown by the rollback in section 7.
- **Consistency:** With a transaction the total stays at **1,500.00**; section 5 showed it can be violated without one.
- **Isolation:** Uncommitted changes are hidden from other transactions.
- **Durability:** Committed changes persist even after a system failure.